In [1]:
# Identifiers (RQx, Hx, Px, Gx, Fx, Tx) are defined in vpir_bench_spec.md: section 2 for the research questions,
# section 11 for the tests behind each gate, section 12 for phases, gates and figures. Rx reproduction targets are
# in docs/reproduction_targets.md, and numbered decisions in plan.md.
# RQ5 (stretch, derived): the cost to re-establish verifiable state after record updates, assuming each update
# triggers the implementation's full rebuild (spec 2, H5). No new runs: every number is a Tier 1 grid median.
import json, sys
from pathlib import Path

import pandas as pd

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.analysis import amortise, frame, plots
from cs4406_sem_project.harness import loadgen, session_check
from cs4406_sem_project.harness.grid import fmt_size

RUNS = [repo / "results/raw/all-20261112_tier1", repo / "results/raw/all-20261112_anchors"]  # notebook 17
platform = json.loads((RUNS[0] / "meta.json").read_text())["platform"]
df, status = frame.load_runs(RUNS, platform)
FIG = repo / "docs/figures"
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)


def save(name, fig):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=200, bbox_inches="tight")


plots._style()
CELL = (128 << 20, 1 << 10)  # decision 28: the RQ4 cell, 128 MiB x 1 KiB (S3 at 1 bit)

t = plots.build(df, status)
med = t.medians
print(f"{len(med)} cells with medians")

99 cells with medians


In [2]:
# Rebuild cost per update = one whole offline phase: server seconds (preprocessing, plus the proof where there is
# one), client seconds (offline verification) and offline bytes (the hint or digest the client downloads again).
# M0 keeps no offline state. 1 KiB records (S3 per bit) at every size each arm ran.
cols = ["server_offline_s", "client_offline_s", "offline_bytes", "server_online_s", "client_online_s", "online_bytes"]
reb = med[(med.record_bytes == 1 << 10) | (med.arm == "S3")][["arm", "db_bytes", "record_bytes", *cols]].copy()
reb["db"] = reb.db_bytes.map(lambda v: fmt_size(int(v)))
print(reb.sort_values(["db_bytes", "arm"])[["arm", "db", *cols]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

   arm     db  server_offline_s  client_offline_s  offline_bytes  server_online_s  client_online_s  online_bytes
    S3   1KiB          0.001053         1.083e-06      4.004e+05        7.417e-06         0.001174          5216
    M0   1MiB                 0          7.08e-07              0        0.0002036        2.479e-06     6.555e+04
    M1   1MiB          0.006848             2e-06             32        0.0002445        7.271e-06      8.66e+04
S0-cpp   1MiB            0.3367                 0      2.228e+07        0.0001865         0.001948     1.088e+04
 S0-go   1MiB           0.07859                 0      7.864e+06        5.833e-05         0.000807          7688
    S1   1MiB             1.347            0.2647      4.867e+07        0.0001845         0.002006     1.088e+04
    S2   1MiB             2.952            0.3819      7.267e+07         0.000401         0.002804     1.626e+04
    S3   1MiB             0.979         5.667e-06      1.275e+07         0.006291          0.038

In [3]:
# H5: per query, at u updates per query (each forcing a rebuild before the next query), the cost is
# online + u x offline, which is amortise's cost at Q = 1 / u. Rank the arms at 128 MiB and 1 GiB x 1 KiB.
import math
US = (0, 0.01, 0.1, 1)
for db in (128 << 20, 1 << 30):
    sel = med[(med.db_bytes == db) & (med.record_bytes == 1 << 10)].set_index("arm")
    for kind in ("total_s", "bytes"):
        fn = amortise.AMORTISED[kind]
        out = pd.DataFrame({f"u={u}": fn(sel, math.inf if u == 0 else 1 / u) for u in US})
        print(f"{fmt_size(db)} x 1 KiB, {kind} per query\n{out.to_string(float_format=lambda v: f'{v:.4g}')}")
        print("most expensive:", {c: out[c].idxmax() for c in out}, "cheapest:", {c: out[c].idxmin() for c in out}, "\n")

128MiB x 1 KiB, total_s per query
           u=0  u=0.01   u=0.1     u=1
arm                                   
M0     0.02529 0.02529 0.02529 0.02529
M1      0.0374 0.04524  0.1158  0.8218
S0-cpp 0.03186  0.3667   3.381   33.52
S0-go  0.01547  0.1184   1.045   10.31
S1     0.03225   1.372   13.43     134
S2     0.04232   1.967   19.28   192.5
most expensive: {'u=0': 'S2', 'u=0.01': 'S2', 'u=0.1': 'S2', 'u=1': 'S2'} cheapest: {'u=0': 'S0-go', 'u=0.01': 'M0', 'u=0.1': 'M0', 'u=1': 'M0'} 

128MiB x 1 KiB, bytes per query
             u=0    u=0.01     u=0.1       u=1
arm                                           
M0     5.244e+05 5.244e+05 5.244e+05 5.244e+05
M1     8.076e+05 8.076e+05 8.076e+05 8.076e+05
S0-cpp 1.092e+05 2.345e+06 2.247e+07 2.237e+08
S0-go   8.67e+04 9.745e+05 8.965e+06 8.887e+07
S1     1.092e+05 5.101e+06 5.002e+07 4.993e+08
S2     1.311e+05 6.163e+06 6.045e+07 6.033e+08
most expensive: {'u=0': 'M1', 'u=0.01': 'S2', 'u=0.1': 'S2', 'u=1': 'S2'} cheapest: {'u=0': 'S0-go'

In [4]:
# The update rate at which rebuilding dominates: u* = online / offline, where the rebuild share of a query's cost
# reaches one half, per arm at each size (client + server seconds). A lower u* means updates hurt sooner.
reb["u_half_time"] = (reb.server_online_s + reb.client_online_s.fillna(0)) / (reb.server_offline_s + reb.client_offline_s.fillna(0))
print(reb.pivot_table(index="arm", columns="db", values="u_half_time").to_string(float_format=lambda v: f"{v:.3g}"))
# The same point as one update every N queries, N = 1 / u*.
print("\nqueries per update at that point (1 / u*):")
print((1 / reb.pivot_table(index="arm", columns="db", values="u_half_time")).to_string(float_format=lambda v: f"{v:,.0f}"))


db       128MiB    16MiB     1GiB  1KiB     1MiB
arm                                             
M0     3.57e+04 4.52e+03 2.71e+05   NaN      291
M1       0.0477   0.0439   0.0457   NaN   0.0368
S0-cpp 0.000951  0.00195 0.000627   NaN  0.00634
S0-go    0.0015  0.00356 0.000955   NaN    0.011
S1     0.000241 0.000475 0.000152   NaN  0.00136
S2      0.00022 0.000395      NaN   NaN 0.000961
S3       0.0179   0.0213      NaN  1.12   0.0458

queries per update at that point (1 / u*):
db      128MiB  16MiB  1GiB  1KiB  1MiB
arm                                    
M0           0      0     0   NaN     0
M1          21     23    22   NaN    27
S0-cpp   1,051    513 1,594   NaN   158
S0-go      666    281 1,047   NaN    91
S1       4,155  2,103 6,593   NaN   736
S2       4,547  2,531   NaN   NaN 1,040
S3          56     47   NaN     1    22
